<a href="https://colab.research.google.com/github/ran4erep/Torrent-Loader/blob/main/Torrent_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 📥 Torrent Downloader { display-mode: "form" }
import os
import re
import subprocess
import shutil
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive

# 1. Mount Google Drive quietly
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Ensure aria2 is installed quietly
status = subprocess.run(['which', 'aria2c'], capture_output=True)
if status.returncode != 0:
    subprocess.run(['apt-get', '-y', 'install', 'aria2'], capture_output=True)

clear_output()

# --- UI Components ---
style = {'description_width': 'initial'}
header = widgets.HTML("<h2>🚀 Torrent Downloader</h2>")

input_url = widgets.Text(description="Magnet/URL:", placeholder="Вставьте ссылку", layout=widgets.Layout(width='98%'), style=style)
file_upload = widgets.FileUpload(accept='.torrent', description="Файл .torrent", layout=widgets.Layout(width='98%'))
input_box = widgets.VBox([widgets.HTML("<b>1. Источник:</b>"), input_url, file_upload], layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='5px'))

btn_start = widgets.Button(description="Начать скачивание", button_style='primary', icon='download', layout=widgets.Layout(width='48%'))
btn_stop = widgets.Button(description="Остановить", button_style='danger', icon='stop', layout=widgets.Layout(width='48%'))
control_box = widgets.HBox([btn_start, btn_stop], layout=widgets.Layout(margin='10px 0'))

prog_bar = widgets.FloatProgress(value=0.0, min=0.0, max=100.0, description='Прогресс:', bar_style='info', layout=widgets.Layout(width='98%'), style=style)
status_label = widgets.HTML("<b>Статус:</b> Готов к работе")
info_label = widgets.HTML("")
progress_box = widgets.VBox([widgets.HTML("<b>2. Состояние:</b>"), status_label, prog_bar, info_label], layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='5px'))

app_ui = widgets.VBox([header, input_box, control_box, progress_box])

current_proc_container = {'process': None}

def update_ui(status_text, progress=0, info=""):
    status_label.value = f"<b>Статус:</b> {status_text}"
    prog_bar.value = progress
    info_label.value = info

def start_download(b):
    local_path = "/content/temp_downloads"
    drive_path = "/content/drive/MyDrive/TorrentDownloads"

    if os.path.exists(local_path): shutil.rmtree(local_path)
    os.makedirs(local_path, exist_ok=True)
    os.makedirs(drive_path, exist_ok=True)

    target = ""
    if input_url.value:
        target = input_url.value
    elif file_upload.value:
        fname = list(file_upload.value.keys())[0]
        local_torrent = os.path.join("/content", fname)
        with open(local_torrent, "wb") as f:
            f.write(file_upload.value[fname]['content'])
        target = local_torrent
    else:
        update_ui("<font color='red'>Ошибка: укажите источник!</font>")
        return

    update_ui("Инициализация...", 0)

    cmd = ['aria2c', '--dir=' + local_path, '--seed-time=0', '--summary-interval=1', '--console-log-level=error', target]
    current_proc_container['process'] = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    for line in current_proc_container['process'].stdout:
        if '(' in line and '%' in line:
            try:
                p_val = int(re.search(r'\((\d+)%\)', line).group(1))
                speed = re.search(r'DL:([^ ]+)', line).group(1)
                eta = re.search(r'ETA:([^\s\]]+)', line).group(1)
                update_ui("Скачивание локально...", p_val, f"Скорость: {speed} | Осталось: {eta}")
            except: pass

    if current_proc_container['process'].wait() == 0:
        update_ui("Перенос на Google Drive...", 100, "Пожалуйста, не закрывайте вкладку...")
        subprocess.run(f"rsync -avP '{local_path}/' '{drive_path}/'", shell=True)
        shutil.rmtree(local_path)
        update_ui("<font color='green'>Готово!</font>", 100, f"Файлы сохранены в Drive/TorrentDownloads")
    else:
        update_ui("<font color='red'>Ошибка загрузки.</font>")

def stop_download(b):
    if current_proc_container['process']:
        current_proc_container['process'].terminate()
        update_ui("Остановлено", 0)

btn_start.on_click(start_download)
btn_stop.on_click(stop_download)
display(app_ui)